In [0]:
import json
import re
from pyspark.sql import functions as F

# SECOP II: carga de Raw a Bronze
- Este notebook registra en tablas Delta los contratos y procesos de **IDARTES** e **IDIPRON** previamente descargados y validados en ADLS Raw.
- Bronze conserva todas las filas y campos del JSON leído, incluidos duplicados y variantes. La limpieza, el tipado de negocio y los cruces se realizan después en Silver.
- Las transformaciones se implementan en PySpark. SQL se utiliza únicamente para inspeccionar esquemas, tipos de tabla y ubicaciones físicas.
- **Origen:** Contenedor `raw/addb01` mediante external locations de Unity Catalog y la identidad administrada del Access Connector.
- **Destino:** Tablas administradas en `project_dev.bronze`, almacenadas en `addb01/lakehouse`.
- **Ejecución:** Desarrollo, con acceso de lectura a Raw y permisos para crear/escribir tablas en Bronze.

## 1. Parámetros de las ingestas auditadas
Los widgets identifican exactamente los snapshots que se van a cargar. No se leen runs antiguos ni descargas parciales.
| Parámetro | Descripción |
| --- | --- |
| run_procesos | Run ID del pipeline padre que descargó los 70 lotes de procesos |
| run_idartes | Run ID de los contratos completos de IDARTES |
| run_idipron | Run ID de los contratos completos de IDIPRON |
| anio | Año de firma de los contratos |

In [0]:
dbutils.widgets.text("run_procesos", "6717455f-416d-4457-9a88-600732d976d6")
dbutils.widgets.text("run_idartes", "95aa87aa-90ee-4104-8f38-d18edc3287b8")
dbutils.widgets.text("run_idipron", "90a6911a-40da-486b-b7c4-9ac9ab37ecfa")
dbutils.widgets.text("anio", "2024")
dbutils.widgets.text("ambiente", "dev", "Ambiente: dev o prod")
dbutils.widgets.text("catalogo", "project_dev", "Catalogo de Unity Catalog")
dbutils.widgets.text("storage_account", "addb01", "Storage account del ambiente")

## 2. Configuración, lectura y trazabilidad
Se valida el formato de los Run IDs y la existencia del esquema `project_dev.bronze`. El notebook no crea ni cambia la ubicación del catálogo.
| Dataset | IDARTES | IDIPRON | Total esperado |
| --- | ---: | ---: | ---: |
| Contratos — jbjy-vk9h | 3651 | 3233 | 6884 |
| Procesos — p6dx-8zbt | 3667 | 2888 | 6555 |

Se leen arrays JSON con multiLine y FAILFAST. primitivesAsString mantiene los valores primitivos como texto para evitar conversiones de negocio prematuras.
Se conservan todas las columnas del origen. Los nombres que comienzan por _ se reservan para trazabilidad; un conflicto detiene la carga.
| Columna agregada | Uso |
| --- | --- |
| _entidad | Identifica IDARTES o IDIPRON a partir de la ruta |
| _run_origen | Identifica la ejecución Raw que produjo la fila |
| _archivo_origen | Ruta completa del archivo JSON |
| _archivo_modificado_utc | Fecha de modificación del archivo de origen |
| _pagina / _lote | Página y lote extraídos del nombre/ruta; contratos no tienen lote y su valor es null |
| _anio_cohorte_contratos | Año de la cohorte contractual, no necesariamente del proceso |
| _dataset_socrata | Identificador del dataset fuente |
| _fecha_ingesta_utc | Momento de carga en Bronze |
| _contenido_json | JSON normalizado de la fila, incluidos valores null |
| _contenido_sha256 | Huella del contenido normalizado para comparar cargas |

El JSON normalizado no es una copia byte a byte del archivo: el original permanece en Raw. El hash no es una clave única de fila, porque los duplicados se conservan. Si una tabla destino ya existe, se exige que sea Delta administrada, esté en lakehouse de desarrollo y tenga las columnas de trazabilidad necesarias.



In [0]:
ambiente = dbutils.widgets.get("ambiente").strip().lower()
catalogo = dbutils.widgets.get("catalogo").strip()
storage_account = dbutils.widgets.get("storage_account").strip()

if ambiente not in {"dev", "prod"}:
    raise ValueError("ambiente debe ser dev o prod")

if not re.fullmatch(r"[A-Za-z_][A-Za-z0-9_]*", catalogo):
    raise ValueError("catalogo debe contener solo letras, numeros y guiones bajos")

if not re.fullmatch(r"[a-z0-9]{3,24}", storage_account):
    raise ValueError(
        "storage_account debe tener entre 3 y 24 letras minusculas o numeros"
    )

if ambiente == "prod" and (
    catalogo.lower() == "project_dev" or storage_account == "addb01"
):
    raise ValueError(
        "Para prod debes indicar su catalogo y storage account; "
        "no se permiten los valores de desarrollo"
    )

raiz_raw = f"abfss://raw@{storage_account}.dfs.core.windows.net/"
raiz_destino = f"abfss://lakehouse@{storage_account}.dfs.core.windows.net/"
esquema_destino = f"{catalogo}.bronze"

print(f"Ambiente: {ambiente}")
print(f"Origen Raw: {raiz_raw}")
print(f"Destino Bronze: {esquema_destino}")
print(f"Raiz de almacenamiento esperada: {raiz_destino}")
anio = int(dbutils.widgets.get("anio"))
if anio != 2024:
    raise ValueError("Este notebook valida la cohorte 2024 ya auditada; no cambiar el anio sin adaptar sus controles")

Ambiente: dev
Origen Raw: abfss://raw@addb01.dfs.core.windows.net/
Destino Bronze: project_dev.bronze
Raiz de almacenamiento esperada: abfss://lakehouse@addb01.dfs.core.windows.net/


In [0]:
def leer_uuid(nombre):
    valor = dbutils.widgets.get(nombre).strip()
    if not re.fullmatch(r"[0-9a-fA-F]{8}(?:-[0-9a-fA-F]{4}){3}-[0-9a-fA-F]{12}", valor):
        raise ValueError(f"UUID invalido: {nombre}")
    return valor

In [0]:
runs = {entidad: leer_uuid(f"run_{entidad}") for entidad in ("idartes", "idipron")}
run_procesos = leer_uuid("run_procesos")
spark.sql(f"DESCRIBE SCHEMA EXTENDED {esquema_destino}").collect()

[Row(database_description_item='Catalog Name', database_description_value='project_dev'),
 Row(database_description_item='Namespace Name', database_description_value='bronze'),
 Row(database_description_item='Comment', database_description_value=''),
 Row(database_description_item='Location', database_description_value=''),
 Row(database_description_item='Owner', database_description_value='lsb.davidbernal@gmail.com'),
 Row(database_description_item='Properties', database_description_value='((unity.catalog.managed.iceberg.defaults.delta.feature.catalogManaged,supported))'),
 Row(database_description_item='Predictive Optimization', database_description_value='ENABLE (inherited from METASTORE my-metastore)')]

In [0]:
configuraciones = {
    "contratos": {
        "dataset": "jbjy-vk9h", "tabla": f"{esquema_destino}.secop_contratos",
        "conteos": {"idartes": 3651, "idipron": 3233},
        "requeridas": {"id_contrato", "proceso_de_compra", "nit_entidad", "fecha_de_firma"},
        "rutas": [raiz_raw + f"secop/pruebas_paginacion/contratos/{e}/anio={anio}/run={runs[e]}/pagina_*.json"
                  for e in ("idartes", "idipron")],
    },
    "procesos": {
        "dataset": "p6dx-8zbt", "tabla": f"{esquema_destino}.secop_procesos",
        "conteos": {"idartes": 3667, "idipron": 2888},
        "requeridas": {"id_del_proceso", "id_del_portafolio"},
        "rutas": [raiz_raw + f"secop/procesos/{e}/anio_contratos={anio}/run={run_procesos}/lote=*/pagina_*.json"
                  for e in ("idartes", "idipron")],
    },
}

In [0]:
def revisar_destino(tabla):
    if not spark.catalog.tableExists(tabla):
        return None
    detalle = spark.sql(f"DESCRIBE DETAIL {tabla}").first().asDict()
    descripcion = spark.sql(f"DESCRIBE TABLE EXTENDED {tabla}").collect()
    tipo = next((r.data_type for r in descripcion if r.col_name.strip() == "Type"), None)
    if tipo != "MANAGED" or detalle["format"] != "delta":
        raise ValueError(f"{tabla}: destino existente no es una tabla Delta administrada")
    if not detalle["location"].startswith(raiz_destino):
        raise ValueError(f"{tabla}: ubicacion fuera de lakehouse de desarrollo: {detalle['location']}")
    necesarias = {"_entidad", "_run_origen", "_archivo_origen", "_contenido_sha256"}
    if not necesarias.issubset(spark.table(tabla).columns):
        raise ValueError(f"{tabla}: tabla previa incompatible; no se modifico")
    return detalle["location"]

In [0]:
def preparar(tipo, config):
    base = (spark.read.option("multiLine", "true").option("mode", "FAILFAST")
            .option("primitivesAsString", "true").json(config["rutas"]))
    if not config["requeridas"].issubset(base.columns):
        raise ValueError(f"{tipo}: faltan campos requeridos")
    if any(c.startswith("_") for c in base.columns):
        raise ValueError(f"{tipo}: campo de origen entra en conflicto con metadatos reservados")
    contenido = F.to_json(F.struct(*[F.col(c) for c in sorted(base.columns)]),
                          options={"ignoreNullFields": "false"})
    df = (base.select("*", F.col("_metadata.file_path").alias("_archivo_origen"),
                      F.col("_metadata.file_modification_time").alias("_archivo_modificado_utc"),
                      contenido.alias("_contenido_json"))
          .withColumn("_contenido_sha256", F.sha2("_contenido_json", 256)))
    segmento = "contratos" if tipo == "contratos" else "procesos"
    df = (df.withColumn("_entidad", F.regexp_extract("_archivo_origen", f"/{segmento}/(idartes|idipron)/", 1))
          .withColumn("_run_origen", F.regexp_extract("_archivo_origen", r"/run=([0-9a-fA-F-]+)/", 1))
          .withColumn("_pagina", F.regexp_extract("_archivo_origen", r"/pagina_(\d+)\.json$", 1).cast("int"))
          .withColumn("_lote", F.regexp_extract("_archivo_origen", r"/lote=(\d+)/", 1).cast("int"))
          .withColumn("_anio_cohorte_contratos", F.lit(anio))
          .withColumn("_dataset_socrata", F.lit(config["dataset"]))
          .withColumn("_fecha_ingesta_utc", F.current_timestamp())
          .cache())
    try:
        conteos = {r['_entidad']: r['count'] for r in df.groupBy("_entidad").count().collect()}
        if conteos != config["conteos"]:
            raise ValueError(f"{tipo}: conteos Raw distintos a los validados: {conteos}")
        for entidad in ("idartes", "idipron"):
            run = runs[entidad] if tipo == "contratos" else run_procesos
            if df.filter((F.col("_entidad") == entidad) & (F.col("_run_origen") != run)).limit(1).count():
                raise ValueError(f"{tipo}/{entidad}: run de origen incorrecto")
        if df.filter(F.col("_pagina").isNull()).limit(1).count():
            raise ValueError(f"{tipo}: numero de pagina invalido")
        if tipo == "procesos" and df.filter(F.col("_lote").isNull()).limit(1).count():
            raise ValueError("Procesos: numero de lote invalido")
        revisar_destino(config["tabla"])
        return df
    except Exception:
        df.unpersist()
        raise

## 3. Validación previa de ambos datasets
Se preparan contratos y procesos antes de empezar a escribir. Un error de conteos, columnas, Run ID, página, lote o destino impide iniciar la carga.
La validación de integridad detallada de Raw se realizó antes con `02_validar_ingesta_procesos`, esta etapa verifica los controles necesarios para su registro en Bronze.

Los conteos de procesos incluyen las 37 repeticiones por contenido de **IDARTES** y todas las variantes con un mismo ID de proceso. No se aplica deduplicación ni se exige una relación uno a uno entre contrato y proceso. Tampoco se descartan contratos sin coincidencia. La caché evita repetir lecturas durante los controles y la carga, se libera ante un error de preparación o al finalizar la escritura.


In [0]:
preparados = {}
try:
    for tipo, config in configuraciones.items():
        preparados[tipo] = preparar(tipo, config)
except Exception:
    for df in preparados.values():
        df.unpersist()
    raise
print("Fuentes y destinos revisados; listos para cargar Bronze.")

Fuentes y destinos revisados; listos para cargar Bronze.


## 4. Carga Delta y reejecución secuencial segura
Para cada dataset y entidad se comprueba si el Run ID ya está registrado en Bronze.
- Si no está cargado, append crea la tabla o agrega las filas sin sobrescribir las existentes. mergeSchema permite incorporar columnas nuevas compatibles.
- Si ya está cargado, se comparan archivo de origen y hash en ambos sentidos mediante exceptAll, conservando la multiplicidad de las filas.
- Si coincide, se omite la escritura y se informa ya_cargado_sin_cambios. Si difiere, el notebook se detiene sin sobrescribir ese run.

Después de cada carga se verifica el conteo de entidad/run y la ubicación física de la tabla. Cada escritura Delta es una transacción, pero las cuatro cargas no forman una transacción conjunta. Si falla después de cargar alguna entidad, la reejecución secuencial reconoce las cargas que ya coinciden.
Esta comprobación no es un bloqueo de concurrencia: no ejecutar dos copias al mismo tiempo. Ambas entidades comparten la tabla de su dataset: secop_contratos y secop_procesos tienen ubicaciones distintas, pero IDARTES e IDIPRON no necesitan tablas físicas separadas.


In [0]:
resumen = []
claves_comparacion = ["_archivo_origen", "_contenido_sha256"]
try:
    for tipo, config in configuraciones.items():
        tabla = config["tabla"]
        df = preparados[tipo]
        for entidad in ("idartes", "idipron"):
            run = runs[entidad] if tipo == "contratos" else run_procesos
            seleccion = df.filter(F.col("_entidad") == entidad)
            cantidad = config["conteos"][entidad]
            existentes = None
            if spark.catalog.tableExists(tabla):
                existentes = spark.table(tabla).filter(
                    (F.col("_entidad") == entidad) & (F.col("_run_origen") == run))
            if existentes is not None and existentes.limit(1).count():
                nuevo = seleccion.select(*claves_comparacion)
                previo = existentes.select(*claves_comparacion)
                # exceptAll conserva multiplicidad: no se eliminan duplicados del origen.
                if nuevo.exceptAll(previo).limit(1).count() or previo.exceptAll(nuevo).limit(1).count():
                    raise ValueError(f"{tabla}/{entidad}: carga previa difiere de Raw; detener, no sobrescribir")
                estado = "ya_cargado_sin_cambios"
            else:
                seleccion.write.format("delta").mode("append").option("mergeSchema", "true").saveAsTable(tabla)
                estado = "cargado"
            ubicacion = revisar_destino(tabla)
            almacenados = spark.table(tabla).filter(
                (F.col("_entidad") == entidad) & (F.col("_run_origen") == run)).count()
            if almacenados != cantidad:
                raise ValueError(f"{tabla}/{entidad}: conteo Bronze incorrecto: {almacenados}")
            resumen.append((tipo, entidad, almacenados, estado, ubicacion))
finally:
    for df in preparados.values():
        df.unpersist()

display(spark.createDataFrame(resumen,
    "dataset string, entidad string, filas long, estado string, ubicacion_delta string").orderBy("dataset", "entidad"))
print("Bronze cargado sin limpiar ni deduplicar; Raw permanece intacto.")

dataset,entidad,filas,estado,ubicacion_delta
contratos,idartes,3651,ya_cargado_sin_cambios,abfss://lakehouse@addb01.dfs.core.windows.net/__unitystorage/catalogs/ab0b04a4-41d3-4906-8c11-9fe611ffcbb4/tables/dc7ad54b-5baa-42a9-8815-4116d4350958
contratos,idipron,3233,ya_cargado_sin_cambios,abfss://lakehouse@addb01.dfs.core.windows.net/__unitystorage/catalogs/ab0b04a4-41d3-4906-8c11-9fe611ffcbb4/tables/dc7ad54b-5baa-42a9-8815-4116d4350958
procesos,idartes,3667,ya_cargado_sin_cambios,abfss://lakehouse@addb01.dfs.core.windows.net/__unitystorage/catalogs/ab0b04a4-41d3-4906-8c11-9fe611ffcbb4/tables/fec6c47a-f4d3-4733-8e21-93408cc8870d
procesos,idipron,2888,ya_cargado_sin_cambios,abfss://lakehouse@addb01.dfs.core.windows.net/__unitystorage/catalogs/ab0b04a4-41d3-4906-8c11-9fe611ffcbb4/tables/fec6c47a-f4d3-4733-8e21-93408cc8870d


Bronze cargado sin limpiar ni deduplicar; Raw permanece intacto.


## 5. Resultado y evidencias
La tabla final muestra dataset, entidad, filas, estado y ubicación Delta. Los estados esperados son `cargado` en la primera carga y `ya_cargado_sin_cambios` al repetirla. Los conteos deben permanecer en 6884 contratos y 6555 filas de procesos. Las rutas deben comenzar por `abfss://lakehouse@addb01.dfs.core.windows.net/`.


In [0]:
dbutils.notebook.exit(json.dumps({"resultados": [
    {"dataset": tipo, "entidad": entidad, "filas": filas, "estado": estado, "ubicacion_delta": ubicacion}
    for tipo, entidad, filas, estado, ubicacion in resumen
]}, ensure_ascii=False))
